In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

df = pd.read_csv('C:\\xampp\\htdocs\\ai-drug-price\\data\\cleaned_data_set\\cleaned.csv')
features = [
     "Generic Name",
    "Strength",
    "Pack Size",
    "Name of the Manufacturer",
    "Dosage Description",
    "Use For"
]
target="Price"
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            features
        )
    ]
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", model)
    ]
)
pipeline.fit(X_train, y_train)


# 2. Save the trained pipeline to disk
model_path = 'C:\\xampp\\htdocs\\ai-drug-price\\models\\medicine_price_model.pkl'
joblib.dump(pipeline, model_path)

print(f"Model successfully trained and saved to: {model_path}")



Training data: (40158, 6)
Testing data: (10040, 6)
Model successfully trained and saved to: C:\xampp\htdocs\ai-drug-price\models\medicine_price_model.pkl


In [6]:
# evaluation of model
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y_test_pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)

print(f"MAE:  {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2:   {r2:.4f}")

MAE:  7.45
MSE: 1415.98
RMSE: 37.63
R2:   0.8443


In [11]:
drug_input = pd.DataFrame(
    [
        {
            "Generic Name": "Paracetamol",
            "Strength": "1000 mg",
            "Pack Size": "1 x 10 Combo Pack",
            "Name of the Manufacturer": "XYZ Pharmasecuicc Ltd.",
            "Dosage Description": "Tablet",
            "Use For": "Human",
        }
    ]
)

drug_pred = pipeline.predict(drug_input)[0]

print(f"\nPredicted Price: {drug_pred:.2f}")


Predicted Price: 2.24


In [17]:
medicine = drug_input.iloc[0]

# 3. Filter dataset for matching market drugs
similar = df[
    (
        df["Generic Name"].astype(str).str.strip().str.lower()
        == str(medicine["Generic Name"]).strip().lower()
    )
    & (
        df["Strength"].astype(str).str.strip().str.lower()
        == str(medicine["Strength"]).strip().lower()
    )
    & (
        df["Dosage Description"].astype(str).str.strip().str.lower()
        == str(medicine["Dosage Description"]).strip().lower()
    )
    & (
        df["Use For"].astype(str).str.strip().str.lower()
        == str(medicine["Use For"]).strip().lower()
    )
].copy()

# 4. Extract market stats if matching drugs exist
if len(similar) > 0:
    market_min = float(similar["Price"].min())
    market_max = float(similar["Price"].max())
    market_median = float(similar["Price"].median())
    max_row = similar.loc[similar["Price"].idxmax()]
else:
    market_min = None
    market_max = None
    market_median = None
    max_row = None

print("market_min:", market_min)
print("market_max:", market_max)
print("market_median:", market_median)

market_min: 1.39
market_max: 2.25
market_median: 2.25
